```{contents}
```

## Label Smoothing


**Label Smoothing** is a regularization technique used in classification models to prevent the network from becoming **overconfident** in its predictions.

In standard supervised learning, the true label is represented as a **one-hot vector**:

| Class | Target Vector |
| ----- | ------------- |
| A     | [1, 0, 0, 0]  |
| B     | [0, 1, 0, 0]  |
| C     | [0, 0, 1, 0]  |

This enforces the model to push the predicted probability of the true class toward **1.0** and all others toward **0.0**, which often leads to:

* Overconfident predictions
* Poor calibration
* Reduced generalization
* Vulnerability to label noise

**Label smoothing** relaxes this constraint.

Instead of assigning probability 1 to the correct class, we assign:

$$
y_{smooth} = (1 - \varepsilon) \cdot y_{true} + \varepsilon / K
$$

Where:

* $\varepsilon$ is the smoothing factor
* $K$ is the number of classes

---

### Intuition

Label smoothing injects **uncertainty** into the learning target:

| Effect                      | Interpretation                                 |
| --------------------------- | ---------------------------------------------- |
| Prevents extreme confidence | The model no longer assumes labels are perfect |
| Improves generalization     | Reduces memorization                           |
| Stabilizes training         | Softens gradients                              |
| Improves calibration        | Predicted probabilities match real confidence  |

It acts similarly to a **Bayesian prior** that discourages certainty.

---

### Workflow

1. Convert hard labels → smoothed labels
2. Compute loss using smoothed targets
3. Backpropagate normally
4. Model learns softer decision boundaries

---

### Mathematical Form

For class $c$:

$$
y_c =
\begin{cases}
1 - \varepsilon & \text{if } c = y \
\varepsilon / (K - 1) & \text{otherwise}
\end{cases}
$$

---

### Remediation and When to Use

| Scenario               | Benefit                      |
| ---------------------- | ---------------------------- |
| Noisy labels           | Reduces overfitting to noise |
| Small datasets         | Improves generalization      |
| Large models           | Prevents overconfidence      |
| Knowledge distillation | Produces softer targets      |

---

### PyTorch Demonstration

#### Without Label Smoothing

```python
import torch
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
```

#### With Label Smoothing (Built-in)

```python
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
```

#### Manual Implementation

```python
def label_smoothing_loss(logits, targets, epsilon, num_classes):
    log_probs = torch.log_softmax(logits, dim=1)
    
    with torch.no_grad():
        smooth_targets = torch.zeros_like(log_probs)
        smooth_targets.fill_(epsilon / (num_classes - 1))
        smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - epsilon)

    loss = (-smooth_targets * log_probs).sum(dim=1).mean()
    return loss
```

---

### Example Training Step

```python
logits = model(inputs)
loss = label_smoothing_loss(logits, labels, epsilon=0.1, num_classes=10)
loss.backward()
```

---

### Variants

| Variant                    | Description                    |
| -------------------------- | ------------------------------ |
| Uniform smoothing          | Same smoothing for all classes |
| Adaptive smoothing         | Larger ε for uncertain samples |
| Confidence-based smoothing | ε depends on model confidence  |
| Distillation smoothing     | Targets from teacher model     |

---

### Empirical Benefits

| Metric            | Impact    |
| ----------------- | --------- |
| Test accuracy     | Improves  |
| Calibration error | Decreases |
| Overfitting       | Reduced   |
| Robustness        | Increased |

---

### Key Takeaways

* Label smoothing is a **simple but powerful regularizer**
* It improves both **generalization** and **calibration**
* It is especially effective for **large neural networks** and **noisy data**
* Easily integrated into modern training pipelines
